# Aprendizado de Máquina — Aula prática 05

## Árvores de Regressão e Ensembles

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

O KNN falha em dimensão alta porque trata
todas as covariáveis igualmente. Precisamos de um método que **escolha em quais
direções olhar**. A árvore de regressão é exatamente isso — e vem com um defeito
enorme de fábrica.

> **A árvore tem viés baixo e variância altíssima. Esta aula inteira é sobre o que
> fazer com essa variância.**

São três respostas, em ordem histórica e em ordem de qualidade: podar (jogar fora
complexidade), ensacar (fazer média de muitas) e impulsionar (construir devagar,
corrigindo o próprio erro). Vamos medir as três, e medir também o **piso** que a
segunda delas não consegue furar — o número que explica por que as florestas
aleatórias existem.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- ler uma árvore nos dois formatos — partição do plano e diagrama — e reconhecer
  onde ela captura uma interação;
- exibir um caso em que a busca **gananciosa** não enxerga a estrutura dos dados;
- podar por custo–complexidade e escolher $\alpha$ por validação cruzada;
- medir quanto a média de muitas árvores melhora a predição, e a partir de
  quantas árvores ela para de melhorar;

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os objetos novos são a árvore e os dois *ensembles* do `scikit-learn`, mais o
`plot_tree`, que desenha a árvore.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. A árvore é uma partição

Uma árvore parte o espaço de covariáveis em retângulos e prevê, em cada um, a
média das respostas de treino que caíram ali. As duas representações — o diagrama
e a partição — são a mesma coisa vista de dois ângulos.

Vamos construir um problema em $[0,10]^2$ com uma **interação**: o efeito de $x_2$
depende de onde está $x_1$. Nenhum modelo linear pega isso sem que alguém escreva
o termo de interação à mão.

In [ ]:
rng = np.random.default_rng(0)
n = 300
X2 = rng.uniform(0, 10, size=(n, 2))


def alvo_2d(M):
    esquerda = M[:, 0] <= 4
    return np.where(esquerda,
                    np.where(M[:, 1] <= 6, 3.0, 4.5),      # a esquerda, corte em 6
                    np.where(M[:, 1] <= 3, 2.5, 0.5))      # a direita, corte em 3


y2 = alvo_2d(X2) + rng.normal(0, 0.35, size=n)

arvore = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X2, y2)
print(f"R^2 no treino: {arvore.score(X2, y2):.3f}")

Antes de olhar o que a árvore encontrou, vale ver o que ela está procurando.

A função `alvo_2d` é **constante por partes em quatro retângulos** — exatamente a
forma que uma árvore sabe representar. A interação anunciada acima está nos
números: o corte de $x_2$ vale $6$ à esquerda de $x_1 = 4$ e $3$ à direita.

O ruído somado tem desvio-padrão $0{,}35$, contra saltos entre regiões vizinhas
que vão de $0{,}5$ a $4{,}0$ — folga suficiente para a árvore reencontrar os
cortes, como a próxima célula mostra.

In [ ]:
gx_r, gy_r = np.meshgrid(np.linspace(0, 10, 300), np.linspace(0, 10, 300))
r_verdade = alvo_2d(np.c_[gx_r.ravel(), gy_r.ravel()]).reshape(gx_r.shape)

fig, ax = subplots(figsize=(5.0, 4.0))
mapa = ax.pcolormesh(gx_r, gy_r, r_verdade, cmap="viridis", shading="auto",
                     vmin=0.5, vmax=4.5)
ax.plot([4, 4], [0, 10], color="white", lw=2)     # o corte de x1
ax.plot([0, 4], [6, 6], color="white", lw=2)      # o corte de x2, a esquerda
ax.plot([4, 10], [3, 3], color="white", lw=2)     # o corte de x2, a direita
for px, py, valor, cor in [(2, 3, 3.0, "white"), (2, 8, 4.5, "black"),
                           (7, 1.5, 2.5, "white"), (7, 6.5, 0.5, "white")]:
    ax.text(px, py, f"{valor}", color=cor, ha="center", va="center", fontsize=11)
ax.set_xlabel("x1"); ax.set_ylabel("x2")
ax.set_title("r(x): a funcao que a arvore vai procurar", fontsize=9)
fig.colorbar(mapa, ax=ax, label="r(x)")

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(10, 3.8),
                           gridspec_kw={"width_ratios": [1, 1.35]})

gx, gy = np.meshgrid(np.linspace(0, 10, 300), np.linspace(0, 10, 300))
Z = arvore.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
ax1.contourf(gx, gy, Z, levels=20, cmap="viridis", alpha=0.65)
ax1.scatter(X2[:, 0], X2[:, 1], c=y2, s=10, cmap="viridis", edgecolor="k", lw=0.2)
ax1.set_xlabel("x1"); ax1.set_ylabel("x2")
ax1.set_title("a particao do plano", fontsize=9)

plot_tree(arvore, ax=ax2, feature_names=["x1", "x2"], filled=True,
          rounded=True, precision=2, fontsize=7)
ax2.set_title("a mesma coisa, como arvore", fontsize=9)

In [ ]:
t = arvore.tree_
print("corte da raiz          : x%d <= %.2f" % (t.feature[0] + 1, t.threshold[0]))
print("corte do filho esquerdo: x%d <= %.2f" % (t.feature[1] + 1, t.threshold[1]))
print("corte do filho direito : x%d <= %.2f" % (t.feature[4] + 1, t.threshold[4]))

Repare no essencial: os dois filhos cortam **a mesma variável $x_2$, em pontos
diferentes**. É assim que uma árvore representa uma interação — sem que ninguém a
tenha especificado, sem termo de produto, sem nada. Foi só deixar cada ramo
escolher o próprio corte.

Daí saem as quatro virtudes estruturais das árvores: são interpretáveis, aceitam
covariáveis categóricas sem *dummies*, fazem seleção de variáveis sozinhas e
capturam interações de graça.

E a desvantagem estrutural, que também é geométrica: **todo corte é perpendicular
a um eixo**. Uma fronteira diagonal como $x_1 + x_2 > 10$ obriga a árvore a
construir uma escadinha, e cada degrau custa dados. Se você suspeita de relação
linear, um modelo linear faz melhor com muito menos.

---
## 3. A miopia do algoritmo ganancioso

Encontrar a árvore ótima é NP-difícil, então o algoritmo escolhe, a cada passo, o
corte que **mais reduz o RSS agora** — e nunca volta atrás. Isso tem um custo, e
dá para exibi-lo com um exemplo de quatro linhas.

Considere $y = \operatorname{sinal}(x_1 x_2)$: vale $+1$ nos quadrantes ímpares e
$-1$ nos pares. **Quatro regiões resolvem o problema exatamente** — e, como não há
ruído nenhum, com RSS zero. Vale ver a cara do problema antes de qualquer conta.

In [ ]:
rng_x = np.random.default_rng(1)
Xx = rng_x.uniform(-1, 1, size=(2000, 2))
yx = np.sign(Xx[:, 0] * Xx[:, 1])
Xx_te = rng_x.uniform(-1, 1, size=(20_000, 2))
yx_te = np.sign(Xx_te[:, 0] * Xx_te[:, 1])

vxx, vyy = np.meshgrid(np.linspace(-1, 1, 400), np.linspace(-1, 1, 400))

fig, (ax1, ax2) = subplots(1, 2, figsize=(8.4, 3.9), constrained_layout=True)

im = ax1.pcolormesh(vxx, vyy, np.sign(vxx * vyy), cmap="coolwarm",
                    shading="auto", vmin=-1, vmax=1)
for sx, sy, s in [(0.5, 0.5, "+1"), (-0.5, 0.5, "-1"),
                  (-0.5, -0.5, "+1"), (0.5, -0.5, "-1")]:
    ax1.text(sx, sy, s, ha="center", va="center", fontsize=15, weight="bold",
             color="white")
ax1.set_title("a funcao: y = sinal(x1 x2)", fontsize=10)

ax2.scatter(Xx[:, 0], Xx[:, 1], c=np.where(yx > 0, "#b2182b", "#2166ac"),
            s=6, alpha=0.65)
ax2.set_title("o que a arvore recebe: 2000 pontos, sem ruido", fontsize=10)

for ax in (ax1, ax2):
    ax.axhline(0, color="black", lw=0.9)
    ax.axvline(0, color="black", lw=0.9)
    ax.set_aspect("equal"); ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
    ax.set_xlabel("x1")
ax1.set_ylabel("x2")
# y so assume dois valores: a barra leva so os dois, para nao sugerir escala
fig.colorbar(im, ax=(ax1, ax2), shrink=0.8, ticks=[-1, 1], label="y")

Um tabuleiro de quatro casas, e uma amostra que o cobre inteiro: nada escondido,
nada de ruído, e a partição que resolve tudo tem quatro peças. É um problema
fácil — **para quem olha as duas covariáveis ao mesmo tempo**.

O algoritmo não olha. Ele escolhe um corte de cada vez, num eixo de cada vez.
Repare no que acontece com o *primeiro*.

In [ ]:
# a referencia de tudo o que vem a seguir: o RSS de quem nao usa covariavel
# nenhuma e chuta a media. Ficar acima dele e' ser pior que nao fazer nada.
rss_media = ((yx_te - yx_te.mean()) ** 2).sum()


def rss_teste(pred):
    return ((yx_te - pred) ** 2).sum()


rss_total = ((yx - yx.mean()) ** 2).sum()
melhor = 0.0
for j in (0, 1):
    for t_corte in np.linspace(-0.9, 0.9, 181):
        esq = Xx[:, j] <= t_corte
        if esq.sum() < 5 or (~esq).sum() < 5:
            continue
        rss = (((yx[esq] - yx[esq].mean()) ** 2).sum()
               + ((yx[~esq] - yx[~esq].mean()) ** 2).sum())
        melhor = max(melhor, rss_total - rss)

print(f"RSS de treino sem nenhum corte : {rss_total:.1f}")
print(f"melhor reducao com UM corte    : {melhor:.4f}  ({100*melhor/rss_total:.3f}%)")

quadrante_tr = (Xx[:, 0] > 0).astype(int) * 2 + (Xx[:, 1] > 0).astype(int)
quadrante_te = (Xx_te[:, 0] > 0).astype(int) * 2 + (Xx_te[:, 1] > 0).astype(int)
medias = np.array([yx[quadrante_tr == q].mean() for q in range(4)])
print(f"\nRSS no teste, chutando a media          : {rss_media:.1f}")
print(f"RSS no teste dos 4 quadrantes (o oraculo): "
      f"{rss_teste(medias[quadrante_te]):.1f}")

In [ ]:
print(f"arvore gananciosa, RSS no TESTE (chutar a media da {rss_media:.0f}):")
for prof in (1, 2, 3, 4, 6, 8, None):
    m = DecisionTreeRegressor(max_depth=prof, random_state=0).fit(Xx, yx)
    rot = "cheia" if prof is None else str(prof)
    r = rss_teste(m.predict(Xx_te))
    print(f"   profundidade {rot:>5}  ({m.get_n_leaves():3d} folhas): {r:8.1f}"
          f"   ({100 * r / rss_media:5.1f}% do RSS da media)")

parada = DecisionTreeRegressor(min_impurity_decrease=0.005, random_state=0).fit(Xx, yx)
print(f"\ncom parada precoce (min_impurity_decrease=0,005): "
      f"{parada.get_n_leaves()} folha(s), RSS = {rss_teste(parada.predict(Xx_te)):.1f}")

A coluna de RSS já diz que a árvore não sai do lugar até a profundidade 4, mas é
difícil sentir o tamanho do problema lendo números. Vale desenhar **o que ela
prevê**, profundidade a profundidade, no mesmo quadrado — com a verdade ao lado
para comparar.

In [ ]:
gxx, gyy = np.meshgrid(np.linspace(-1, 1, 250), np.linspace(-1, 1, 250))
grade = np.c_[gxx.ravel(), gyy.ravel()]

fig, eixos = subplots(2, 4, figsize=(11, 5.8), constrained_layout=True)


def painel(ax, Z, titulo):
    im = ax.pcolormesh(gxx, gyy, Z.reshape(gxx.shape), cmap="coolwarm",
                       shading="auto", vmin=-1, vmax=1)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(titulo, fontsize=8)
    return im


im = painel(eixos[0, 0], np.sign(grade[:, 0] * grade[:, 1]),
            "a verdade: sinal(x1 x2)\n4 regioes, RSS 0")
for ax, prof in zip(eixos.ravel()[1:7], (1, 2, 3, 4, 6, 8)):
    m = DecisionTreeRegressor(max_depth=prof, random_state=0).fit(Xx, yx)
    painel(ax, m.predict(grade), f"profundidade {prof}\n{m.get_n_leaves()} folhas, "
                                 f"RSS {rss_teste(m.predict(Xx_te)):.0f}")
painel(eixos[1, 3], parada.predict(grade),
       f"parada precoce\n{parada.get_n_leaves()} folha, "
       f"RSS {rss_teste(parada.predict(Xx_te)):.0f}")
fig.colorbar(im, ax=eixos, shrink=0.55, label="predicao")
fig.suptitle("a predicao da arvore: a proximidade da realidade só é alcançada na profundidade 8",
             fontsize=10)

O melhor primeiro corte não reduz **nada**: qualquer reta vertical ou horizontal
deixa os dois lados com média zero. E é isso que a última linha da tabela mostra:
um critério de parada do tipo *"só divida se a redução for relevante"* — um
`min_impurity_decrease` modestíssimo — **mata a árvore na raiz**, deixando-a com
uma única folha e RSS de $20005$, o de quem não olha covariável nenhuma, num
problema que quatro folhas resolvem com RSS **zero**.

A figura mostra o resto, e é mais eloquente que a tabela. Nas profundidades 1 a 4
o quadrado é **cinza**: a árvore prevê quase zero em toda parte, e o RSS fica em
torno de $19500$, colado no $20000$ de chutar a média — na profundidade 1 ela fica
até **pior**, com $20123$. As faixas coloridas nas bordas são os cortes que ela de
fato fez: existem, e não servem para nada. Na 6 o tabuleiro começa a aparecer,
ainda pálido, e o RSS cai para $6884$. Só na 8 ela alcança o oráculo, com $264$ —
e gasta **12 folhas**, o triplo do necessário, porque cada corte mal colocado no
topo tem de ser remendado lá embaixo.

Duas consequências práticas, e as duas organizam o resto da aula:

1. **Cresça demais e pode depois**, em vez de parar cedo. A poda pode reconhecer
   *a posteriori* que sobrava estrutura; um critério de parada precoce decide antes
   de ter visto o que havia adiante.
2. **Uma árvore só não é o produto final.** Se a decisão no topo é praticamente um
   empate, ela é decidida por ruído.

Em duas dimensões, quatro regiões resolviam o problema e o primeiro corte não via
nada. Em três, oito regiões resolvem — e a pergunta é se a miopia piora ou melhora
quando há mais um eixo para o algoritmo tropeçar.

Não dá para desenhar $[-1,1]^3$ num plano, mas duas fatias bastam para ver o que
muda: o tabuleiro **inverte** quando $x_3$ troca de sinal. É por isso que os
octantes se cancelam dois a dois em qualquer corte.

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(7.2, 3.4))
for ax, x3 in ((ax1, -0.5), (ax2, 0.5)):
    Z = np.sign(grade[:, 0] * grade[:, 1] * x3).reshape(gxx.shape)
    ax.pcolormesh(gxx, gyy, Z, cmap="coolwarm", shading="auto", vmin=-1, vmax=1)
    ax.set_aspect("equal"); ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.set_title(f"fatia x3 = {x3}", fontsize=9)
fig.suptitle("sinal(x1 x2 x3): o tabuleiro se inverte ao trocar o sinal de x3",
             fontsize=9)
fig.tight_layout()

In [ ]:
rng_3 = np.random.default_rng(1)
X3 = rng_3.uniform(-1, 1, size=(2000, 3))
y3 = np.sign(X3[:, 0] * X3[:, 1] * X3[:, 2])
X3_te = rng_3.uniform(-1, 1, size=(20_000, 3))
y3_te = np.sign(X3_te[:, 0] * X3_te[:, 1] * X3_te[:, 2])

rss_media3 = ((y3_te - y3_te.mean()) ** 2).sum()


def rss_3d(pred):
    return ((y3_te - pred) ** 2).sum()


def melhor_corte(Xc, yc):
    """Varre todos os cortes de todas as colunas do no.

    Devolve (RSS do no, reducao do melhor corte, coluna, limiar) -- o limiar
    tambem, para podermos DESCER pelo corte que o algoritmo de fato escolheu.
    """
    rss_no = ((yc - yc.mean()) ** 2).sum()
    melhor = (0.0, None, None)
    for j in range(Xc.shape[1]):
        for t in np.linspace(-0.9, 0.9, 181):
            esq = Xc[:, j] <= t
            if esq.sum() < 5 or (~esq).sum() < 5:
                continue
            rss = (((yc[esq] - yc[esq].mean()) ** 2).sum()
                   + ((yc[~esq] - yc[~esq].mean()) ** 2).sum())
            if rss_no - rss > melhor[0]:
                melhor = (rss_no - rss, j, t)
    return (rss_no, *melhor)


print("os dois primeiros niveis do algoritmo ganancioso, no por no:\n")
print(f"  {'no':16s}{'n':>6s}{'RSS do no':>11s}{'melhor corte':>16s}"
      f"{'reduz':>9s}{'% do RSS do no':>16s}")

rss_raiz, red_raiz, j_raiz, t_raiz = melhor_corte(X3, y3)
print(f"  {'raiz':16s}{len(y3):6d}{rss_raiz:11.1f}"
      f"{f'x{j_raiz + 1} <= {t_raiz:+.3f}':>16s}{red_raiz:9.4f}"
      f"{100 * red_raiz / rss_raiz:15.3f}%")

esquerda = X3[:, j_raiz] <= t_raiz      # os dois filhos do corte que ELE escolheu
for nome, mask in [("filho esquerdo", esquerda), ("filho direito", ~esquerda)]:
    rss_no, red, j, t = melhor_corte(X3[mask], y3[mask])
    print(f"  {nome:16s}{mask.sum():6d}{rss_no:11.1f}"
          f"{f'x{j + 1} <= {t:+.3f}':>16s}{red:9.4f}{100 * red / rss_no:15.3f}%")

oitante = ((X3[:, 0] > 0).astype(int) * 4 + (X3[:, 1] > 0).astype(int) * 2
           + (X3[:, 2] > 0).astype(int))
rss_oraculo = sum(((y3[oitante == q] - y3[oitante == q].mean()) ** 2).sum()
                  for q in range(8))
print(f"\n  para comparar, a particao verdadeira em 8 octantes deixa RSS {rss_oraculo:.1f}")

In [ ]:
print(f"arvore gananciosa, RSS no TESTE (chutar a media da {rss_media3:.0f}):")
for prof in (1, 2, 3, 4, 5, 6, 8, None):
    m3 = DecisionTreeRegressor(max_depth=prof, random_state=0).fit(X3, y3)
    rot = "cheia" if prof is None else str(prof)
    r = rss_3d(m3.predict(X3_te))
    print(f"   profundidade {rot:>5}  ({m3.get_n_leaves():3d} folhas): {r:8.1f}"
          f"   ({100 * r / rss_media3:5.1f}% do RSS da media)")

parada3 = DecisionTreeRegressor(min_impurity_decrease=0.005, random_state=0).fit(X3, y3)
print(f"\ncom parada precoce: {parada3.get_n_leaves()} folha(s), "
      f"RSS = {rss_3d(parada3.predict(X3_te)):.1f}")

As duas tabelas contam a mesma história em escalas diferentes de profundidade.
Vale pô-las no mesmo eixo, com a linha de quem chuta a média por referência: tudo
o que estiver acima dela é uma árvore que custou folhas para ficar pior que não
fazer nada.

In [ ]:
profs = list(range(1, 11))
rss_2d = [rss_teste(DecisionTreeRegressor(max_depth=k, random_state=0)
                    .fit(Xx, yx).predict(Xx_te)) for k in profs]
rss_3d_prof = [rss_3d(DecisionTreeRegressor(max_depth=k, random_state=0)
                      .fit(X3, y3).predict(X3_te)) for k in profs]

fig, ax = subplots(figsize=(6.4, 3.6))
ax.plot(profs, rss_2d, "o-", color="steelblue", ms=4,
        label="2D: sinal(x1 x2), 4 regioes bastam")
ax.plot(profs, rss_3d_prof, "s-", color="crimson", ms=4,
        label="3D: sinal(x1 x2 x3), 8 regioes bastam")
ax.axhline(rss_media, ls="--", color="gray", lw=1.2)
ax.text(1.3, rss_media + 700, "chutar a media", fontsize=7.5, color="gray")
ax.axhline(0, ls=":", color="green", lw=1.2)
ax.text(8.4, 900, "o oraculo", fontsize=7.5, color="green")
ax.set_xlabel("profundidade maxima da arvore"); ax.set_ylabel("RSS no teste")
ax.set_xticks(profs); ax.set_xlim(0.6, 10.4); ax.set_ylim(-900, 22500)
ax.set_title("acima da linha cinza a arvore e' pior que nao fazer nada", fontsize=9)
ax.legend(fontsize=7.5, loc="lower left")

**Piora, e piora de todos os jeitos.**

Leia a tabela dos nós pela **última coluna**. O melhor corte possível na raiz reduz
$0{,}194\%$ do RSS dali, e nos dois filhos que esse corte produz os melhores cortes
reduzem $0{,}447\%$ e $0{,}942\%$ — a cegueira **não** é só do primeiro passo. Para
ter escala: a partição verdadeira em oito octantes deixa RSS **zero**. Os dois
primeiros níveis do algoritmo encontram menos de $1\%$ do que havia para encontrar.

As outras colunas enganam se lidas isoladamente. O RSS cai de $1998$ na raiz para
$1414$ no filho, mas não porque o corte tenha explicado alguma coisa — é só que
sobraram menos observações. Como $y=\pm 1$ e a média local fica sempre perto de
zero, **o RSS de um nó é praticamente o seu número de observações**, e por isso a
porcentagem é a única coluna que compara nós de tamanhos diferentes.

E uma ressalva, porque a tentação é imediata: essa porcentagem **não** é o que o
`min_impurity_decrease` compara. O `scikit-learn` pondera pelo tamanho do nó, e o
que ele confronta com o limiar é a redução dividida pelo total de observações de
treino — $0{,}0019$, $0{,}0032$ e $0{,}0027$ nas três linhas. As três reprovam em
$0{,}005$, e é isso que a próxima célula confirma.

A consequência de tudo está na tabela de profundidades e na figura. Oito regiões
resolvem o problema exatamente, o que caberia numa árvore de profundidade 3. Mas
até a profundidade 5 a árvore gananciosa tem RSS **acima** do de chutar a média, e
pior a cada nível: $20111$, $20304$, $20392$, $20563$, $20703$. Ela não está apenas
deixando de aprender — está gastando folhas para ficar mais errada que o preditor
constante. Na profundidade 6 cai para $16576$, e só a árvore cheia, com 59 folhas,
chega a $2652$, $13\%$ do RSS da média.

E a parada precoce, que no caso de duas dimensões já era perigosa, aqui é fatal:
com `min_impurity_decrease=0,005` a árvore fica com **uma folha**. O critério
reprova já na raiz, e nenhum dos cortes que viriam depois chega a ser considerado.

É por isso que a poda por complexidade de custo, que cresce a árvore inteira e
*depois* corta, é o método padrão — e não a parada precoce.

---
## 4. Crescer e podar

A poda por custo–complexidade minimiza

$$\sum_{k}\sum_{i:\,X_i \in R_k}(Y_i - \widehat y_{R_k})^2 + \alpha\,|T|,$$

onde $|T|$ é o número de folhas. É a mesma estrutura da Aula 02 — ajuste mais
$\lambda \times$ complexidade — com o número de folhas no papel da norma do vetor
de coeficientes.

O `scikit-learn` entrega a sequência inteira de $\alpha$ em que a árvore ótima
muda, de graça.

In [ ]:
rng_p = np.random.default_rng(5)
n_p, d_p = 400, 6


def alvo_p(M):
    return np.sin(1.5 * M[:, 0]) + 0.8 * M[:, 1] * M[:, 2] + 0.5 * M[:, 0] ** 2


X_p = rng_p.uniform(-2, 2, size=(n_p, d_p))
y_p = alvo_p(X_p) + rng_p.normal(0, 1.0, size=n_p)
X_pte = rng_p.uniform(-2, 2, size=(4000, d_p))
r_pte = alvo_p(X_pte)

cheia = DecisionTreeRegressor(random_state=0).fit(X_p, y_p)
caminho = cheia.cost_complexity_pruning_path(X_p, y_p)
alphas = caminho.ccp_alphas[:-1]          # o ultimo colapsa para a raiz
print(f"a arvore cheia tem {cheia.get_n_leaves()} folhas")
print(f"o caminho de poda tem {len(alphas)} valores de alpha, "
      f"de {alphas.min():.5f} a {alphas.max():.3f}")

In [ ]:
folhas, risco_te = [], []
for a in alphas:
    m = DecisionTreeRegressor(ccp_alpha=a, random_state=0).fit(X_p, y_p)
    folhas.append(m.get_n_leaves())
    risco_te.append(np.mean((m.predict(X_pte) - r_pte) ** 2))

busca = skm.GridSearchCV(DecisionTreeRegressor(random_state=0),
                         {"ccp_alpha": alphas},
                         cv=skm.KFold(5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error").fit(X_p, y_p)
a_cv = busca.best_params_["ccp_alpha"]
m_cv = DecisionTreeRegressor(ccp_alpha=a_cv, random_state=0).fit(X_p, y_p)

fig, (ax1, ax2) = subplots(1, 2, figsize=(7.8, 3.0))
ax1.plot(alphas, folhas, drawstyle="steps-post", color="steelblue")
ax1.axvline(a_cv, ls=":", color="green")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("alpha"); ax1.set_ylabel("numero de folhas")
ax1.set_title("podar é andar para a direita", fontsize=9)

ax2.plot(alphas, risco_te, color="crimson")
ax2.axvline(a_cv, ls=":", color="green", label="alpha escolhido por CV")
ax2.set_xscale("log")
ax2.set_xlabel("alpha"); ax2.set_ylabel("risco (teste)")
ax2.set_title("e existe um alpha ótimo", fontsize=9)
ax2.legend(fontsize=8)

print(f"alpha escolhido por CV : {a_cv:.4f}")
print(f"folhas depois da poda  : {m_cv.get_n_leaves()}  (eram {cheia.get_n_leaves()})")
print(f"risco da arvore cheia  : {np.mean((cheia.predict(X_pte) - r_pte) ** 2):.4f}")
print(f"risco da arvore podada : {np.mean((m_cv.predict(X_pte) - r_pte) ** 2):.4f}")

Os números dizem que a poda ajudou — o risco caiu quase pela metade. Mas eles não
mostram **o que** foi podado, e é aí que o método fica intuitivo. Vale desenhar as
duas árvores lado a lado.

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

plot_tree(cheia, ax=ax1, impurity=False, filled=True, precision=1)
ax1.set_title(f"a arvore cheia: {cheia.get_n_leaves()} folhas, profundidade "
              f"{cheia.get_depth()}", fontsize=10)

plot_tree(m_cv, ax=ax2, impurity=False, filled=True, precision=1, fontsize=8,
          feature_names=[f"x{j + 1}" for j in range(d_p)])
ax2.set_title(f"depois da poda (alpha = {a_cv:.3f}): {m_cv.get_n_leaves()} folhas, "
              f"profundidade {m_cv.get_depth()}", fontsize=10)

usadas = sorted({f"x{f + 1}" for f in m_cv.tree_.feature if f >= 0})
print(f"cortes da arvore podada, todos em: {usadas}")
print(f"a verdade e sin(1.5 x1) + 0.8 x2 x3 + 0.5 x1^2")

À esquerda não se lê nada, e é esse o ponto: são **400 folhas para 400
observações** — uma por folha. Ela acerta cada ponto de treino exatamente, com erro
de treino zero, e é a **pior** das duas contra a verdade. À direita, a mesma árvore
depois de a poda passar: 5 folhas, profundidade 4, e o risco caindo de $2{,}59$
para $1{,}29$.

Repare em que covariáveis os quatro cortes sobreviventes olham: **todos em $x_1$**.
A verdade é $\operatorname{sen}(1{,}5x_1) + 0{,}8\,x_2x_3 + 0{,}5x_1^2$, e a
árvore podada não tem um único corte em $x_2$ ou $x_3$ — a interação inteira ficou
de fora. Não é que ela seja irrelevante: é que com cinco folhas não há como
representar uma interação, e o que sobra é o efeito marginal mais forte.

Dá para ver o preço disso, porque conhecemos $r$: ela depende de **três** das seis
covariáveis, e as duas parcelas se desenham separadamente.

In [ ]:
fig, (ax1, ax2, ax3) = subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)

# --- o eixo que a arvore enxerga: r ao longo de x1, com o resto fixo em zero
t = np.linspace(-2, 2, 400)
L = np.zeros((400, d_p)); L[:, 0] = t
ax1.plot(t, alvo_p(L), color="green", lw=2, label="r verdadeira")
ax1.plot(t, cheia.predict(L), color="lightsteelblue", lw=1, label="cheia (400 folhas)")
ax1.plot(t, m_cv.predict(L), color="crimson", lw=1.8, label="podada (5 folhas)")
ax1.set_xlabel("x1"); ax1.set_ylabel("r")
ax1.set_title("ao longo de x1 (x2..x6 = 0)", fontsize=10)
ax1.legend(fontsize=7)

# --- o plano que ela nao enxerga: a interacao 0.8 x2 x3, com x1 fixo em zero
g2, g3 = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
G = np.zeros((g2.size, d_p)); G[:, 1] = g2.ravel(); G[:, 2] = g3.ravel()
const = np.unique(m_cv.predict(G))

for ax, Z, titulo in [(ax2, alvo_p(G), "a verdade no plano x2-x3 (x1 = 0)"),
                      (ax3, m_cv.predict(G),
                       f"o que a podada preve ali: {const[0]:.2f}, em todo lugar")]:
    im = ax.pcolormesh(g2, g3, Z.reshape(g2.shape), cmap="viridis",
                       shading="auto", vmin=-3.2, vmax=3.2)
    ax.set_aspect("equal"); ax.set_xlabel("x2"); ax.set_title(titulo, fontsize=10)
ax2.set_ylabel("x3")
fig.colorbar(im, ax=(ax2, ax3), shrink=0.85, label="r")

print(f"a podada assume {len(const)} valor(es) no plano x2-x3")
print(f"a verdade ali varia de {alvo_p(G).min():.2f} a {alvo_p(G).max():.2f}")

**À esquerda, a parte que a árvore pegou.** A curva verde é
$\operatorname{sen}(1{,}5x_1)+0{,}5x_1^2$, e a escada vermelha de cinco degraus a
acompanha decentemente — é o melhor que cinco folhas conseguem fazer com uma
função lisa. A linha azul-clara é a árvore cheia no mesmo corte: ela oscila entre
$-2$ e $+4$ onde a verdade é suave. A variância do painel anterior, agora com cara.

**No meio e à direita, a parte que ela não pegou.** A verdade no plano
$x_2$–$x_3$ é a sela $0{,}8\,x_2x_3$, que vai de $-3{,}2$ a $+3{,}2$. A árvore
podada prevê ali **um único número**, $0{,}63$, em todo lugar — o painel da direita
é de uma cor só. Não é aproximação ruim: é ausência de aproximação, porque nenhum
corte dela olha $x_2$ ou $x_3$.

E note que os dois painéis da direita usam a **mesma** escala de cor. É por isso
que dá para comparar os dois de relance: o que a árvore entrega no lugar da sela é
o valor médio, plano.

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(8.4, 3.9), sharey=True,
                          constrained_layout=True)

for ax, m, rotulo in [(ax1, cheia, f"cheia, {cheia.get_n_leaves()} folhas"),
                      (ax2, m_cv, f"podada, {m_cv.get_n_leaves()} folhas")]:
    pred = m.predict(X_pte)
    ax.scatter(r_pte, pred, s=4, alpha=0.25, color="steelblue")
    lim = [r_pte.min() - 0.4, r_pte.max() + 0.4]
    ax.plot(lim, lim, color="black", lw=1)      # onde cairia a predicao perfeita
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("r(x) verdadeiro")
    ax.set_title(f"{rotulo} — risco {np.mean((pred - r_pte) ** 2):.2f}", fontsize=10)
ax1.set_ylabel("predicao")

for nome, m in [("cheia", cheia), ("podada", m_cv)]:
    print(f"valores distintos que a arvore {nome:6s} prediz nos 4000 pontos: "
          f"{len(np.unique(m.predict(X_pte)))}")

A árvore cheia produz 389 valores distintos nos 4000 pontos de teste; a podada,
**cinco**. São as cinco faixas horizontais — horizontais porque dentro de cada
folha a predição é constante por construção, não importa o quanto $r$ varie ali.

E os dois painéis erram de maneiras opostas. A cheia acompanha a diagonal em média,
mas com uma dispersão enorme: é **variância**. A podada é certeira e grossa — pouca
dispersão, e cinco degraus para aproximar uma superfície lisa: é **viés**. Nenhuma
das duas é boa, e nenhum $\alpha$ conserta as duas coisas ao mesmo tempo, porque o
botão é um só.

É esse impasse que a próxima seção ataca, e por outro caminho: em vez de procurar a
árvore certa, tirar a média de muitas árvores erradas.

---
## 5. Por que a média ajuda

A Seção 4 terminou num impasse: a árvore cheia erra por variância, a podada erra
por viés, e o $\alpha$ é um botão só — não dá para consertar as duas coisas ao
mesmo tempo.

A saída é não escolher. Em vez de procurar *a* árvore certa, cresça **muitas**
árvores diferentes e tire a média. Cada uma continua ruim; o que melhora é o
conjunto. Duas receitas para fabricar essa diferença:

- ***bagging***: cada árvore vê uma amostra *bootstrap* dos dados, e nada mais muda;
- **floresta aleatória**: o mesmo, e além disso cada nó só pode escolher entre
  $m < p$ covariáveis sorteadas.

A pergunta prática é uma só: quanto isso melhora a predição, e a partir de quantas
árvores para de melhorar? Basta medir.

In [ ]:
def risco_p(modelo):
    return np.mean((modelo.predict(X_pte) - r_pte) ** 2)


Bs = [1, 2, 5, 10, 25, 50, 100, 300]
linhas = []
for B in Bs:
    bag = BaggingRegressor(DecisionTreeRegressor(random_state=0), n_estimators=B,
                           random_state=0, n_jobs=-1).fit(X_p, y_p)
    # m = 3 aqui; escolher o m e' assunto da Secao 7
    flo = RandomForestRegressor(n_estimators=B, max_features=3,
                                random_state=0, n_jobs=-1).fit(X_p, y_p)
    linhas.append({"B": B, "bagging": risco_p(bag), "floresta": risco_p(flo)})

tab_ens = pd.DataFrame(linhas).set_index("B")
print(f"para comparar, uma arvore so:  podada {risco_p(m_cv):.4f}   "
      f"cheia {risco_p(cheia):.4f}\n")
print(tab_ens.round(4).to_string())

In [ ]:
fig, ax = subplots(figsize=(6.0, 3.6), constrained_layout=True)

ax.axhline(risco_p(m_cv), color="green", ls="--", lw=1.4,
           label=f"uma arvore podada ({risco_p(m_cv):.2f})")
ax.axhline(risco_p(cheia), color="gray", ls=":", lw=1.4,
           label=f"uma arvore cheia ({risco_p(cheia):.2f})")
ax.plot(tab_ens.index, tab_ens["bagging"], "o-", color="crimson", ms=5,
        label="bagging")
ax.plot(tab_ens.index, tab_ens["floresta"], "s-", color="steelblue", ms=5,
        label="floresta (m = 3)")

ax.set_xscale("log")
ax.set_xlabel("B (numero de arvores)")
ax.set_ylabel("risco no teste")
ax.set_title("a media de muitas arvores ruins bate qualquer arvore boa", fontsize=10)
ax.legend(fontsize=8)

Três leituras, e a primeira é contraintuitiva.

**Uma árvore do *ensemble*, sozinha, é pior que a árvore podada.** Em $B=1$ o
*bagging* dá $2{,}97$ e a floresta $2{,}84$, contra $1{,}29$ da podada da Seção 4.
Faz sentido: essas árvores crescem sem poda e cada uma vê só uma amostra
*bootstrap* — cerca de 63% dos dados distintos. São piores de propósito.

**Mas a média delas bate qualquer árvore isolada, e com folga.** Com $B=25$ o
*bagging* já está em $0{,}81$, contra $1{,}29$ da melhor árvore única: o risco cai
**36%** abaixo do que a poda mais cuidadosa conseguiu. É o ponto inteiro do
capítulo — não se conserta uma árvore, troca-se ela por muitas.

**E o ganho acaba cedo.** De $B=1$ a $B=25$ o risco cai por um fator de quase
quatro; de $B=25$ a $B=300$ ele oscila na terceira casa e não melhora mais.
Acrescentar árvores além disso é gastar computação à toa, mas não faz mal nenhum.

Entre as duas receitas a diferença é pequena neste problema: em $B=300$ a floresta
fica em $0{,}79$ e o *bagging* em $0{,}84$. O $m=3$ que usamos aqui não caiu do céu
— é o que a Seção 6 vai justificar, varrendo os valores possíveis.

---
## 6. Escolhendo $m$

O parâmetro que separa a floresta do *bagging* é o `max_features`. A regra empírica
é $m \approx p/3$ em regressão (e $\sqrt p$ em classificação), mas é um
hiperparâmetro como qualquer outro. Vamos varrer os valores possíveis e olhar o
risco de cada um.

In [ ]:
linhas = []
for m in [1, 2, 3, 4, 6]:
    rf = RandomForestRegressor(n_estimators=300, max_features=m,
                               random_state=0, n_jobs=-1).fit(X_p, y_p)
    linhas.append({"m": m, "m/p": m / d_p,
                   "risco (teste)": np.mean((rf.predict(X_pte) - r_pte) ** 2)})
pd.DataFrame(linhas).set_index("m").round(4)

O mínimo fica no meio, como a teoria manda: $m$ pequeno demais deixa as árvores
bem diferentes entre si, mas estraga cada uma delas; $m = p$ é o *bagging*, em que
elas se parecem demais. A regra $m \approx p/3$ não é lei — aqui o melhor
foi $m = p/2$ — mas cai perto o bastante para servir de ponto de partida.


Com muito lixo na lista, sortear poucas variáveis por nó aumenta a chance de
nenhuma delas prestar. Acrescentamos 10 colunas de puro ruído, indo de $p=6$ para
$p=16$, e varremos $m$ nos dois casos.

In [ ]:
for extras in (0, 10):
    rng_m = np.random.default_rng(5)
    d_tot = d_p + extras
    Xm = rng_m.uniform(-2, 2, size=(n_p, d_tot))
    ym = alvo_p(Xm) + rng_m.normal(0, 1.0, size=n_p)
    Xm_te = rng_m.uniform(-2, 2, size=(4000, d_tot))
    rm_te = alvo_p(Xm_te)

    linhas_m = []
    for m in [1, 2, 3, 4, 6, 8, 12, 16]:
        if m > d_tot:
            continue
        rf_m = RandomForestRegressor(n_estimators=300, max_features=m,
                                     random_state=0, n_jobs=-1).fit(Xm, ym)
        linhas_m.append({"m": m, "m/p": m / d_tot,
                         "risco (teste)": np.mean((rf_m.predict(Xm_te) - rm_te) ** 2)})
    t = pd.DataFrame(linhas_m).set_index("m")
    print(f"p = {d_tot}  ({extras} colunas irrelevantes)   m* = {t['risco (teste)'].idxmin()}")
    print(t.round(4).to_string())
    print()

**O $m$ ótimo se desloca para a direita, e bastante:** de $m^*=3$ com $p=6$ para
$m^*=12$ com $p=16$. Em fração das colunas, de $0{,}50$ para $0{,}75$ — bem acima
do $p/3$ da regra empírica, que daria $2$ e $5$.

O mecanismo é o que o enunciado antecipava. Com $p=16$ e só 3 colunas úteis,
sortear $m=1$ dá $3/16$ de chance de o nó ter alguma variável que preste; o resto
do tempo a árvore corta em ruído. O risco com $m=1$ é $1{,}53$, contra $0{,}99$ no
ótimo — 55% pior.

Vale notar o que **não** mudou: em ambos os casos, usar todas as colunas ($m=p$) é
pior que o ótimo. Mesmo afogada em lixo, a floresta ainda quer alguma
descorrelação entre as árvores; ela só quer menos. O `max_features` continua sendo
um botão a girar, e a regra $p/3$ é um chute inicial, não uma resposta — o que
fica mais evidente justamente quando há colunas irrelevantes, que é a situação
comum fora do livro.


---
## 7. Os três no `superconductivity.csv`

Fechamos com a comparação completa, no conjunto que já usamos nas Aulas 02 a 04:
21.263 materiais, 81 atributos, temperatura crítica como resposta.

In [ ]:
import os

_nome = "superconductivity.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

df = pd.read_csv(_caminho)
Xs = df.drop(columns="critical_temp")
ys = df["critical_temp"].values

rng_s = np.random.default_rng(0)
sub = rng_s.choice(len(ys), size=6000, replace=False)
X_tr, X_te, y_tr, y_te = skm.train_test_split(Xs.values[sub], ys[sub],
                                              test_size=0.3, random_state=0)
print("treino:", X_tr.shape, "  teste:", X_te.shape)

In [ ]:
import time

modelos = {
    "Ridge (Aula 02)": Pipeline([("escala", StandardScaler()),
                                 ("ridge", skl.Ridge(alpha=1.0))]),
    "arvore podada": DecisionTreeRegressor(ccp_alpha=1.0, random_state=0),
    "bagging (B=300)": BaggingRegressor(DecisionTreeRegressor(random_state=0),
                                        n_estimators=300, random_state=0, n_jobs=-1),
    "floresta (B=300, m=d/3)": RandomForestRegressor(n_estimators=300,
                                                     max_features=1/3,
                                                     random_state=0, n_jobs=-1),
}

linhas = []
for nome, m in modelos.items():
    t0 = time.perf_counter()
    m.fit(X_tr, y_tr)
    linhas.append({"modelo": nome,
                   "EQM no teste": mean_squared_error(y_te, m.predict(X_te)),
                   "segundos": time.perf_counter() - t0})
linhas.append({"modelo": "chutar a media", "EQM no teste": y_te.var(), "segundos": 0.0})
pd.DataFrame(linhas).set_index("modelo").round(3)

A distância entre o modelo linear e os *ensembles* é enorme: a Ridge fica em mais
do dobro do erro da floresta. Duas observações sobre o resto.

A **árvore isolada** perde para tudo — mas ganha da Ridge, o que já diz alguma
coisa sobre a não linearidade deste problema. Ela é matéria-prima, não produto
final.

O **bagging e a floresta empatam**: as 81 colunas
são fortemente redundantes, então não existe *uma* covariável dominante para todas
as árvores agarrarem — e o sorteio de covariáveis da floresta tem pouco o que
diversificar que o *bootstrap* já não tenha diversificado.


---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| árvore = partição | §2 | os dois ramos cortam $x_2$ em pontos diferentes: interação de graça |
| cortes nos eixos | §2 | fronteira diagonal vira escadinha, e cada degrau custa dados |
| miopia gananciosa | §3 | 4 folhas resolvem o XOR e a gananciosa gasta 12; uma parada precoce a mata na raiz |
| miopia em 3D | §3 | com um eixo a mais o RSS **sobe** da profundidade 1 à 5 ($20111 \to 20703$): folhas gastas para ficar pior que chutar a média |
| poda | §4 | `cost_complexity_pruning_path` dá a sequência inteira; $\alpha$ sai por CV |
| a média de árvores | §5 | uma árvore do *ensemble* é pior que a podada (2,97 × 1,29), e a média de 25 delas é bem melhor (0,81) |
| quantas árvores | §5 | o ganho acaba em $B \approx 25$; daí em diante é computação à toa, mas inofensiva |
| escolher $m$ | §6 | o ótimo fica no meio e **anda para a direita** com colunas irrelevantes: de $m^*=3$ com $p=6$ para $m^*=12$ com $p=16$ |
| caso real | §7 | floresta e *bagging* empatam na frente, e ambos batem a Ridge com folga |

**Leitura recomendada.** [AME] §4.8 (árvores) e §4.9 (*bagging* e florestas).
[ISLP] §8.1 (crescer e podar, com a mesma figura de partição da Seção 2),
§8.2.1–8.2.2 (*bagging*) e §8.2.3 (florestas).

**Para praticar.** `Lista teorica 05.pdf` (teórica, com gabarito) e
`Lista prática 05.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** A Aula 06 arruma a casa: `Pipeline`, `ColumnTransformer` e a
disciplina que impede que a padronização, a seleção de variáveis ou a imputação
vazem informação do teste para o treino. Sem ela, todos os números deste notebook
seriam suspeitos.